In [ ]:
"""The code is originally from https://nb.bohrium.dp.tech/detail/3413343451


'The code is originally from https://nb.bohrium.dp.tech/detail/3413343451\n\n1. the use of vectroize(u) is removed, it is not necessary for the code to work, also it is slow\n'

### problem 1 settings 

In [1]:
import numpy as np
import math
import time
from numpy.linalg import solve
import matplotlib.pyplot as plt
from scipy.linalg import lstsq
import torch 
if torch.cuda.is_available():  
    device = "cuda" 
else:  
    device = "cpu" 
def PiecewiseGQ1D_weights_points(x_l,x_r,Nx, order):
    """ Output the coeffients and weights for piecewise Gauss Quadrature 
    Parameters
    ----------
    x_l : float 
    left endpoint of an interval 
    
    x_r: float
    right endpoint of an interval 
    
    integration_intervals: int
    number of subintervals for integration
    
    Returns
    -------
    coef1_expand
    
    gw_expand
    
    integration_points
    """
    x,w = np.polynomial.legendre.leggauss(order)
    gx = torch.tensor(x).to(device)
    gx = gx.view(1,-1) # row vector 
    gw = torch.tensor(w).to(device)    
    gw = gw.view(-1,1) # Column vector 
    nodes = torch.linspace(x_l,x_r,Nx+1).view(-1,1).to(device) 
    coef1 = ((nodes[1:,:] - nodes[:-1,:])/2) # n by 1  
    coef2 = ((nodes[1:,:] + nodes[:-1,:])/2) # n by 1  
    coef2_expand = coef2.expand(-1,gx.size(1)) # Expand to n by p shape, -1: keep the first dimension n , expand the 2nd dim (columns)
    integration_points = coef1@gx + coef2_expand
    integration_points = integration_points.flatten().view(-1,1) # Make it a column vector
    gw_expand = torch.tile(gw,(Nx,1)) # rows: n copies of current tensor, columns: 1 copy, no change
    # Modify coef1 to be compatible with func_values
    coef1_expand = coef1.expand(coef1.size(0),gx.size(1))    
    coef1_expand = coef1_expand.flatten().view(-1,1)

    return coef1_expand.to(device)*gw_expand.to(device), integration_points.to(device)


## test 1. 

vanal_f = np.vectorize(f) is not faster. 

In [2]:
def error_plot(multi_Errors):
    plt.figure(figsize=[7, 5])
    plt.tick_params(labelsize=10)
    font2 = {
    'weight' : 'normal',
    'size'   : 22,
    }
    plt.xlabel('Degrees of freedom',font2)
    plt.ylabel('$L_2$ absolute error',font2)
    plt.xscale('log')
    plt.yscale('log')
    # Label = ['FDM','PINN','RFM']
    Label = ['RFM']
    for i in range(len(multi_Errors)):
        Error = multi_Errors[i]
        plt.plot(Error[:,0], Error[:,1], \
                 lw=1.5, ls='-', clip_on=False,\
                 marker='o', markersize=10, \
                 label = Label[i],\
                 markerfacecolor='none',\
                 markeredgewidth=1.5)
    plt.legend()
    plt.title("Comparison of accuracy on 1D Helmholtz equation")
    plt.show()

def time_plot(multi_Errors):
    plt.figure(figsize=[7, 5])
    plt.tick_params(labelsize=10)
    font2 = {
    'weight' : 'normal',
    'size'   : 22,
    }
    plt.xlabel('Degrees of freedom',font2)
    plt.ylabel('Solving time',font2)
    Label = ['FDM','PINN','RFM']
    for i in range(len(multi_Errors)):
        Error = multi_Errors[i]
        plt.plot(Error[:,0], Error[:,2], \
                 lw=1.5, ls='-', clip_on=False,\
                 marker='o', markersize=10, \
                 label = Label[i],\
                 markerfacecolor='none',\
                 markeredgewidth=1.5)
    plt.legend()
    plt.title("Comparison of efficiency on 1D Helmholtz equation")
    plt.show()
    

## RFM

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
torch.set_default_dtype(torch.float64)

R_m = 2.0 
## original tests 
def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        if activation == 'tanh':
            nn.init.uniform_(m.weight, a = -R_m, b = R_m)
            nn.init.uniform_(m.bias, a = -R_m, b = R_m)
        if activation == 'relu':
            nn.init.uniform_(m.weight, a = 1, b = 1)
            nn.init.uniform_(m.bias, a = -1 , b = 1.2)
            nodes = -m.bias.data.squeeze()/m.weight.data.squeeze()
            if len(nodes[nodes < -1]) == 0: 
                print("all nodes are within the interval")
                m.bias.data[0] = 1.2 

# network definition
class Network(nn.Module):
    def __init__(self, d, M):
        super(Network, self).__init__()
        self.fc_layer = nn.Sequential(nn.Linear(d, M, bias=True),nn.Tanh())
        self.output_layer = nn.Linear(M, 1, bias = False)
        
    def forward(self, x):
        h = self.fc_layer(x)
        out = self.output_layer(h)
        return out

In [4]:
R_m = 1 
w = torch.arange(40).double()
b = torch.arange(40).double() 
nn.init.uniform_(w, a = 1, b = 1)
nn.init.uniform_(b, a = -R_m, b = R_m+0.1)

print(-b/w)

tensor([ 0.2635, -0.2510,  0.1137, -1.0498,  0.0156, -0.7909,  0.4083,  0.3730,
        -0.0891, -0.4170, -0.0402,  0.2727, -0.6610,  0.3602, -0.9689,  0.3790,
         0.1752, -0.3443, -0.9004,  0.3284, -0.8171,  0.5759,  0.1757,  0.9097,
        -0.6954,  0.8734, -1.0537,  0.2376,  0.0666, -0.6720,  0.5579,  0.2322,
        -0.7793,  0.9258,  0.1827, -0.9774, -0.5919, -0.0047, -0.3547,  0.7405])


### 

explanations: 
1. $\sigma(\omega \cdot \tilde{x} +b )$ for each subdomain, $\tilde{x}$ is properly scaled 
2. store the models on each subdomain in a list 

In [5]:
# computational domain
X_min = 0.0
X_max = 8.0

# # random initialization for parameters
# def weights_init(m):
#     if isinstance(m, (nn.Conv2d, nn.Linear)):
#         nn.init.uniform_(m.weight, a = -R_m, b = R_m)
#         nn.init.uniform_(m.bias, a = -R_m, b = R_m)

class ReLUk(nn.Module):
    def __init__(self, k):
        super(ReLUk, self).__init__()
        self.k = k

    def forward(self, x):
        return torch.relu(x)**self.k 
    
class Gaussian(nn.Module):
    def __init__(self):
        super(Gaussian, self).__init__()

    def forward(self, x):
        return torch.exp(-x**2)
    
class cosine_activation(nn.Module): 
    def __init__(self):
        super(cosine_activation, self).__init__()
    def forward(self, x):
        return torch.cos(x)
    
class RFM_rep_a(nn.Module): # assume defined on [-1,1], with proper scaling, it can be used for any interval 
    def __init__(self, d, J_n, x_min, x_max, activation = 'tanh', k = 1 ):
        super(RFM_rep_a, self).__init__()
        self.d = d
        self.J_n = J_n
        self.r = (x_max - x_min) / 2.0

        self.x_c = (x_max + x_min)/2
        self.activation = activation 
        self.k = k 
        if self.activation == 'tanh':
            self.phi = nn.Sequential(nn.Linear(self.d, self.J_n, bias=True),nn.Tanh())
        if self.activation == 'relu':
            self.phi = nn.Sequential(nn.Linear(self.d, self.J_n, bias=True),ReLUk(self.k))
        if self.activation == 'sigmoid':
            self.phi = nn.Sequential(nn.Linear(self.d, self.J_n, bias=True),nn.Sigmoid())
        if self.activation == 'gaussian':
            self.phi = nn.Sequential(nn.Linear(self.d, self.J_n, bias=True),Gaussian()) 
        if self.activation == 'cosine': 
            self.phi = nn.Sequential(nn.Linear(self.d, self.J_n, bias=True),cosine_activation()) 

    def forward(self,x):
        x = (x - self.x_c) / self.r # this normalizatoin is important. allows a same random distribution to be used for all regions 
        # print(self.r)
        x = self.phi(x)
        return x


# random feature basis when using \psi^{b} as PoU function
class RFM_rep_b(nn.Module):
    def __init__(self, d, J_n, x_max, x_min):
        super(RFM_rep_b, self).__init__()
        self.d = d
        self.J_n = J_n
        self.n = x_min/(X_max-X_min) * M_p
        self.r = (x_max - x_min) / 2.0
        self.x_0 = (x_max + x_min)/2
        self.phi = nn.Sequential(nn.Linear(self.d, self.J_n, bias=True),nn.Tanh())

    def forward(self,x):
        d = (x - self.x_0) / self.r
        psi = ((d <= -3/4) & (d > -5/4)) * (1+torch.sin(2*np.pi*d))/2 + ((d <= 3/4) & (d > -3/4)) * 1.0 + ((d <= 5/4) & (d > 3/4)) * (1-torch.sin(2*np.pi*d))/2
        y = self.phi(d)
        if self.n == 0:
            psi = ((d <= 3/4) & (d > -1)) * 1.0 + ((d <= 5/4) & (d > 3/4)) * (1-torch.sin(2*np.pi*d))/2
        elif self.n == M_p-1:
            psi = ((d <= -3/4) & (d > -5/4)) * (1+torch.sin(2*np.pi*d))/2 + ((d <= 1) & (d > -3/4)) * 1.0
        else:
            psi = ((d <= -3/4) & (d > -5/4)) * (1+torch.sin(2*np.pi*d))/2 + ((d <= 3/4) & (d > -3/4)) * 1.0 + ((d <= 5/4) & (d > 3/4)) * (1-torch.sin(2*np.pi*d))/2
        return(psi*y)

# predefine the random feature functions in each PoU region
def pre_define(M_p,J_n,activation,k):
    models = []
    for n in range(M_p):
        x_min = (X_max-X_min)/M_p * n + X_min
        x_max = (X_max-X_min)/M_p * (n+1) + X_min
        model = RFM_rep_a(d = 1, J_n = J_n, x_min = x_min, x_max = x_max, activation=activation, k = k)
        model = model.apply(weights_init)
        # print(model.phi[0].weight)
        # print(model.phi[0].bias)
        # nodes = -model.phi[0].bias.data.squeeze()/model.phi[0].weight.data.squeeze()
        # print(nodes)
        model = model.double()
        # x = torch.tensor([0.1])
        # print(model(x)) 
        for param in model.parameters():
            param.requires_grad = False
        models.append(model)
    return(models)

In [6]:
# calculate the l^{infty}-norm and l^{2}-norm error for u
def test(models,M_p,J_n,Q,w,plot = False, print_res = True):
    epsilon = []
    true_values = []
    numerical_values = []
    test_Q = 2*Q
    for n in range(M_p):
        points = torch.tensor(np.linspace((X_max-X_min)/M_p * n + X_min, (X_max-X_min)/M_p * (n+1) + X_min, test_Q+1),requires_grad=False).reshape([-1,1])
        out = models[n](points)
        values = out.detach().numpy()
        numerical_value = np.dot(values, w[n*J_n:(n+1)*J_n,:]) 
        true_value = u(points.numpy()).reshape([-1,1])
        numerical_values.extend(numerical_value) # append
        true_values.extend(true_value)
        epsilon.extend(true_value - numerical_value)
    true_values = np.array(true_values)
    numerical_values = np.array(numerical_values)
    epsilon = np.array(epsilon)
    epsilon = np.maximum(epsilon, -epsilon)
    if print_res: 
        print('R_m=%s,M_p=%s,J_n=%s,Q=%s'%(R_m,M_p,J_n,Q))
        print('L_infty error =',epsilon.max(),', L_2 error =',math.sqrt(8*sum(epsilon*epsilon)/len(epsilon)))
    x = [((X_max - X_min)/M_p)*i / test_Q  for i in range(M_p*(test_Q+1))]
    return(math.sqrt((X_max-X_min)*sum(epsilon*epsilon)/len(epsilon)))



Collocation points. $M_p$ subdomains 
1. interior points: $M_p \times Q$
2. boundary points: 2 
3. smoothness conditions: $(M_p - 1 ) \times 2$
Design matrix is of size: # of data points $\times $ # of dofs (features) 
Each subdomain corresponds to a model. All models are stored in a list. 

Main function:
1. points are pre-allocated in list 

In [ ]:
# Assembling the matrix A,f in linear system 'Au=f'
def assemble_matrix(models,points,M_p,J_n,Q,lamb):
    """This is the data matrix. Each row of A is a data point constraint. 
    """
    A_I = np.zeros([M_p*Q, M_p*J_n]) # PDE term
    A_B = np.zeros([2, M_p*J_n]) # boundary condition
    A_C_0 = np.zeros([M_p-1, M_p*J_n]) # 0-order smoothness condition
    A_C_1 = np.zeros([M_p-1, M_p*J_n]) # 1-order smoothness condition
    f = np.zeros([M_p*Q + 2*(M_p - 1) + 2, 1])
    
    for n in range(M_p):
        # forward and grad
        point = torch.tensor(points[n], requires_grad=True)
        out = models[n](point) # points * <J_n>
        # if n == 0:
            # print("output")
            # print(out)
        values = out.detach().numpy()
        value_l, value_r = values[0,:], values[-1,:] # cell boundary values 
        grad1 = []
        grad2 = []
        for i in range(J_n):
            g1 = torch.autograd.grad(outputs=out[:,i], inputs=point,
                                  grad_outputs=torch.ones_like(out[:,i]),
                                  create_graph = True, retain_graph = True)[0] # dim: points * 1 
            grad1.append(g1.squeeze().detach().numpy()) # dim: features * points
            
            g2 = torch.autograd.grad(outputs=g1[:,0], inputs=point,
                                  grad_outputs=torch.ones_like(out[:,i]),
                                  create_graph = False, retain_graph = True)[0]
            grad2.append(g2.squeeze().detach().numpy()) # dim: features * points 
        grad1 = np.array(grad1).T  # dim: points * features 
        grad2 = np.array(grad2).T
        grad_l = grad1[0,:] # feature evaluations on the first point (left)
        grad_r = grad1[-1,:] # feature evaluations on the last point (right)

        Lu = - grad2 + lamb * values # -u_xx + lamb * u 

        
        # Lu = f condition
        A_I[n*Q:(n + 1)*Q, n*J_n:(n + 1)*J_n] = Lu[:Q,:]
        f[n*Q:(n + 1)*Q, :] = F(points[n], lamb).reshape([-1,1])[:Q]
        
        # boundary conditions, two points 
        if n == 0:
            A_B[0, :J_n] = value_l 
        if n == M_p-1:
            A_B[1, -J_n:] = value_r
        
        # smoothness conditions 
        if M_p > 1:
            if n == 0 : #first region 
                A_C_0[0, :J_n] = -value_r
                A_C_1[0, :J_n] = -grad_r
            elif n == M_p - 1: # last gregion 
                A_C_0[M_p - 2, -J_n:] = value_l
                A_C_1[M_p - 2, -J_n:] = grad_l
            else: # regions in between
                A_C_0[n-1,n*J_n:(n + 1)*J_n] = value_l
                A_C_1[n-1,n*J_n:(n + 1)*J_n] = grad_l
                A_C_0[n,n*J_n:(n + 1)*J_n] = -value_r
                A_C_1[n,n*J_n:(n + 1)*J_n] = -grad_r
    if M_p > 1:
        A = np.concatenate((A_I,A_B,A_C_0,A_C_1),axis=0)
    else:
        A = np.concatenate((A_I,A_B),axis=0)
    
    # boundary conditions
    f[M_p*Q,:] = u(0.)
    f[M_p*Q+1,:] = u(8.)
    return(A,f)

def main(M_p, J_n, Q, lamb, activation, k, print_res = True):
    # prepare collocation points
    time_begin = time.time()
    points = []
    for n in range(M_p):
        x_min = (X_max-X_min)/M_p * n + X_min
        x_max = (X_max-X_min)/M_p * (n+1) + X_min
        points.append(np.linspace(x_min, x_max, Q+1).reshape([-1,1]))
    # prepare models
    models = pre_define(M_p,J_n,activation, k)
    # print(models)
    # model = models[0]
    # print( - model.phi[0].bias.data.squeeze()/model.phi[0].weight.data.squeeze())    

    # matrix define (Au=f)
    A,f = assemble_matrix(models, points, M_p, J_n, Q, lamb)
    # print("matrix")
    # print(A)
    if print_res:
        print('***********************')
        print('Matrix shape: N=%s,M=%s'%(A.shape))

    ## debug the matrix 

    # for i in range(len(A)):
    #     m = abs(A[i,:]).max()  
    #     if abs(m)< 1e-12: 
    #         print("row %s is zero"%(i))


    # rescaling
    c = 100.0
    for i in range(len(A)):
        ## change 
        max_a = abs(A[i,:]).max()
        max_b = A[i,:].max()
        if max_a != max_b: 
            ratio = -c/max_a
            A[i,:] = A[i,:]*ratio
            f[i] = f[i]*ratio
        else: 
            ratio = c/max_a
            A[i,:] = A[i,:]*ratio
            f[i] = f[i]*ratio
    
    # solve
    w = lstsq(A,f)[0]
    
    # test
    error = test(models,M_p,J_n,Q,w, print_res = print_res)
    
    time_end = time.time()
    return models,w, error, time_end - time_begin 

### original parameters

In [8]:
# analytical solution parameters
AA = 1
aa = 8.0*np.pi
bb = 8.0*np.pi
lamb = 4
activation = 'tanh'
k = 3 # if using tanh, k is not used 

# def u(x):
#     return AA * np.sin(bb * (x + 0.05)) * np.cos(aa * (x + 0.05)) + 2.0

# def d2u_dx2(x):
#     return -AA*(aa*aa+bb*bb)*np.sin(bb*(x+0.05))*np.cos(aa*(x+0.05))\
#            -2.0*AA*aa*bb*np.cos(bb*(x+0.05))*np.sin(aa*(x+0.05))

# def F(points, lamb):
#     return(- d2u_dx2(points)  + lamb * u(points)) # change 

def u(x):
    return AA *  np.cos(aa * (x + 0.05)) + 2.0

def d2u_dx2(x):
    return -AA*(aa*aa) * np.cos(aa*(x+0.05)) 

def F(points, lamb):
    return(- d2u_dx2(points)  + lamb * u(points)) # change 


# x_coord = np.linspace(0, 8, 1000).reshape(-1, 1) 
# u_values = u(x_coord)
# plt.plot(x_coord, u_values)
# plt.show() 


def evaluate_model(models,M_p,J_n,w,x_coord,plot = False, print_res = True):
    """
    Need M_p, J_n, w to evaluate the model on x_coord 
    """
    numerical_values = []
    for n in range(M_p):
        # points = torch.tensor(np.linspace((X_max-X_min)/M_p * n + X_min, (X_max-X_min)/M_p * (n+1) + X_min, test_Q+1),requires_grad=False).reshape([-1,1])
        x_min = (X_max-X_min)/M_p * n + X_min
        x_max = (X_max-X_min)/M_p * (n+1) + X_min
        if n==0:
            points = torch.tensor(x_coord[(x_coord >= x_min) & (x_coord <= x_max)], requires_grad=False).reshape([-1,1]) 
        else: 
            points = torch.tensor(x_coord[(x_coord > x_min) & (x_coord <= x_max)], requires_grad=False).reshape([-1,1])
        out = models[n](points)
        values = out.detach().numpy()
        numerical_value = np.dot(values, w[n*J_n:(n+1)*J_n,:]) 
        true_value = u(points.numpy()).reshape([-1,1])
        numerical_values.append(numerical_value) # append
    numerical_values = np.array(numerical_values)
    numerical_values = np.concatenate(numerical_values, axis = 0) 
    return numerical_values 




In [9]:
## original tests 
def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        if activation != 'relu': # Not ReLU 
            nn.init.uniform_(m.weight, a = -R_m, b = R_m)
            nn.init.uniform_(m.bias, a = -R_m, b = R_m)
        if activation == 'relu':
            nn.init.uniform_(m.weight, a = 1, b = 1)
            nn.init.uniform_(m.bias, a = -1 , b = 1.2)
            nodes = -m.bias.data.squeeze()/m.weight.data.squeeze()
            if len(nodes[nodes < -1]) == 0: 
                print("all nodes are within the interval")
                m.bias.data[0] = 1.2 

if __name__ == '__main__':
    lamb = 4
    PoU_nums = 5 
    activation = 'tanh'
    k = 2 # if using tanh, k is not used 
    num_trials = 1 

    for R_m in [0.5,1,2 ,4,8,16]: # 0.5,1,2 
        print() 
        J_n = 50 # the number of basis functions per PoU region
        Q = 50 # the number of collocation pointss per PoU regio
        RFM_Error = np.zeros([PoU_nums,3])
        err_list = np.zeros([PoU_nums,num_trials])
        for i in range(PoU_nums): # the number of PoU regions
            for trial in range(num_trials): 
                M_p =  2 * (2**i)
                RFM_Error[i,0] = int(M_p * J_n)
                models,w, RFM_Error[i,1], RFM_Error[i,2] = main(M_p,J_n,Q,lamb,activation, k, print_res = False)
                err_list[i,trial] = RFM_Error[i,1] 
        print(err_list.mean(axis = 1)) 
        # error_plot([RFM_Error])
        # time_plot([RFM_Error])

/var/folders/45/p1_q2drx75983lk7ph_1gx6mhpr5_w/T/ipykernel_33385/4274200674.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return(math.sqrt((X_max-X_min)*sum(epsilon*epsilon)/len(epsilon)))


[1.08504625e+01 1.26215467e+02 1.82735087e+00 2.10877385e-05
 4.59796012e-09]

[8.81750593e+01 4.39418986e+02 7.36244684e-04 6.05655110e-08
 2.85800157e-10]

[1.35744315e+04 1.05773022e+01 8.41681243e-05 5.12237703e-08
 5.58489853e-10]

[1.18974511e+05 4.82237906e+02 2.48552877e-02 1.07845654e-03
 3.96226939e-06]

[1.41254001e+04 1.81345071e+04 2.72962112e+00 2.32278302e+02
 6.68884043e-02]

[7.55312357e+06 9.10880707e+03 4.72579945e+05 1.19999646e+03
 5.23979194e+02]


In [ ]:
def output_results_Rs_latex(err_list_Rs, R_m_list, num_diff_features_list):
    num_rows = len(num_diff_features_list)
    num_cols = len(R_m_list)

    # Start LaTeX table
    latex_str = "\\begin{tabular}{c|" + "c" * num_cols + "}\n"
    
    # Header row: R values
    latex_str += "\#Features $\\backslash$ R"
    for R in R_m_list:
        latex_str += f" & {R}"
    latex_str += " \\\\\n\\hline\n"

    # Data rows: each row is for a fixed number of features
    for i, n_feat in enumerate(num_diff_features_list):
        latex_str += f"{n_feat}"
        for j in range(num_cols):
            err = err_list_Rs[j][i]  # error for R = R_m_list[j], features = num_diff_features_list[i]
            latex_str += f" & {err:.2e}"
        latex_str += " \\\\\n"

    latex_str += "\\end{tabular}"
    return latex_str

import numpy as np

R_m_list = [1, 2, 3]
num_diff_features_list = [10, 20, 30]
err_list_Rs_test = [
    np.array([1e-2, 5e-3, 1e-3]),   # errors for R = 1
    np.array([8e-3, 4e-3, 9e-4]),   # errors for R = 2
    np.array([6e-3, 3e-3, 7e-4])    # errors for R = 3
]

print(output_results_Rs_latex(err_list_Rs_test, R_m_list, num_diff_features_list))


tensor(170.6667)
170.66666666666666


### Get the average error

In [ ]:
# analytical solution parameters
AA = 1
aa = 8.0*np.pi
bb = 8.0*np.pi
lamb = 4
activation = 'tanh'
k = 3 # if using tanh, k is not used 

def u(x):
    return AA * np.sin(bb * (x + 0.05)) * np.cos(aa * (x + 0.05)) + 2.0

def d2u_dx2(x):
    return -AA*(aa*aa+bb*bb)*np.sin(bb*(x+0.05))*np.cos(aa*(x+0.05))\
           -2.0*AA*aa*bb*np.cos(bb*(x+0.05))*np.sin(aa*(x+0.05))

def F(points, lamb):
    return(- d2u_dx2(points)  + lamb * u(points)) # change 

# def u(x):
#     return AA *  np.cos(aa * (x + 0.05)) + 2.0

# def d2u_dx2(x):
#     return -AA*(aa*aa) * np.cos(aa*(x+0.05)) 

# def F(points, lamb):
#     return(- d2u_dx2(points)  + lamb * u(points)) # change 
if __name__ == '__main__':
    lamb = 4
     
    J_n = 50 # the number of basis functions per PoU region
    
    print_res = False 
    num_diff_features = 5 
    num_trials = 1  
    RFM_Error = np.zeros([num_diff_features,3]) 
    err_list_trials =  np.zeros([num_diff_features,num_trials])
    err_list_Rs = [] 
    activation = 'tanh'
    relu_k = 3 
    
    R_m_list = [0.75, 1.5,3, 6,9,12]
    for R_m in R_m_list: 
        print("Uinsg R_m = ", R_m)
        print("========================================") 
        for i in range(num_diff_features): # the number of PoU regions
            for trial in range(num_trials):
                M_p = 2 * (2**i)
                Q = int(3200/M_p) # the number of collocation pointss per PoU region
                RFM_Error[i,0] = int(M_p * J_n)
                models,w,RFM_Error[i,1], RFM_Error[i,2] = main(M_p,J_n,Q,lamb,activation=activation, k = relu_k, print_res= print_res)
                err_list_trials[i,trial] = RFM_Error[i,1] 

        print(err_list_trials.mean(axis=1)) 
        err_list_Rs.append(err_list_trials.mean(axis=1)) 

R_m_list = [0.75, 1.5,3, 6,9,12]
num_diff_features_list = [ 100,200,400,800,1600]
table_latex = output_results_Rs_latex(err_list_Rs, R_m_list, num_diff_features_list)
print(table_latex)

Uinsg R_m =  0.75


/var/folders/45/p1_q2drx75983lk7ph_1gx6mhpr5_w/T/ipykernel_33385/4274200674.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return(math.sqrt((X_max-X_min)*sum(epsilon*epsilon)/len(epsilon)))


[4.69957145e+00 4.89661417e+01 8.56988511e+01 1.99087664e-02
 4.94542544e-07]
Uinsg R_m =  1.5
[3.29252836e+00 2.64706070e+01 2.98395978e+00 4.68005166e-06
 2.10012275e-09]
Uinsg R_m =  3
[9.07854211e+00 8.20528283e+01 9.38623184e-04 1.17694622e-06
 4.35140338e-09]
Uinsg R_m =  6
[7.01000220e+00 2.13891317e+02 9.22662283e-02 1.16883005e-03
 3.00633746e-05]
Uinsg R_m =  9
[1.13856400e+01 4.47241654e+01 2.81448011e+00 1.50906802e-02
 3.41689478e-03]
Uinsg R_m =  12
[17.10771437 20.40383809  6.75118605  0.06089257  0.13894682]
\begin{tabular}{c|cccccc}
#Features $\backslash$ R & 0.75 & 1.5 & 3 & 6 & 9 & 12 \\
\hline
100 & 4.70e+00 & 3.29e+00 & 9.08e+00 & 7.01e+00 & 1.14e+01 & 1.71e+01 \\
200 & 4.90e+01 & 2.65e+01 & 8.21e+01 & 2.14e+02 & 4.47e+01 & 2.04e+01 \\
400 & 8.57e+01 & 2.98e+00 & 9.39e-04 & 9.23e-02 & 2.81e+00 & 6.75e+00 \\
800 & 1.99e-02 & 4.68e-06 & 1.18e-06 & 1.17e-03 & 1.51e-02 & 6.09e-02 \\
1600 & 4.95e-07 & 2.10e-09 & 4.35e-09 & 3.01e-05 & 3.42e-03 & 1.39e-01 \\
\end{tabular}

### multi-scale frequency test 

In [22]:
# analytical solution parameters
AA = 1
a1 = 2.0*np.pi
a2 = 8.0*np.pi
a3 = 16.0*np.pi
lamb = 4
activation = 'tanh'
k = 3 # if using tanh, k is not used 

def u(x):
    return AA * np.sin(a1 * x) + 0.5 * AA * np.sin(a2 * x) + 0.5 * AA * np.sin(a3 * x) + 2.0 

def d2u_dx2(x):
    return -AA*(a1*a1)*np.sin(a1*x) - 0.5*AA*(a2*a2)*np.sin(a2*x) - 0.5*AA*(a3*a3)*np.sin(a3*x)

def F(points, lamb):
    return(- d2u_dx2(points)  + lamb * u(points)) # change 


if __name__ == '__main__':
    lamb = 4
     
    J_n = 50 # the number of basis functions per PoU region
    
    print_res = False 
    num_diff_features = 5 
    num_trials = 1  
    RFM_Error = np.zeros([num_diff_features,3]) 
    err_list_trials =  np.zeros([num_diff_features,num_trials])
    err_list_Rs = [] 
    activation = 'tanh'
    relu_k = 3 
    
    R_m_list = [0.75, 1.5,3, 6,9,12]
    for R_m in R_m_list: 
        print("Uinsg R_m = ", R_m)
        print("========================================") 
        for i in range(num_diff_features): # the number of PoU regions
            for trial in range(num_trials):
                M_p = 2 * (2**i)
                Q = int(3200/M_p) # the number of collocation pointss per PoU region
                RFM_Error[i,0] = int(M_p * J_n)
                models,w,RFM_Error[i,1], RFM_Error[i,2] = main(M_p,J_n,Q,lamb,activation=activation, k = relu_k, print_res= print_res)
                err_list_trials[i,trial] = RFM_Error[i,1] 

        print(err_list_trials.mean(axis=1)) 
        err_list_Rs.append(err_list_trials.mean(axis=1)) 


    # error_plot([RFM_Error])
    # time_plot([RFM_Error])


Uinsg R_m =  0.75


/var/folders/45/p1_q2drx75983lk7ph_1gx6mhpr5_w/T/ipykernel_33385/4274200674.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return(math.sqrt((X_max-X_min)*sum(epsilon*epsilon)/len(epsilon)))


[4.06830438e+00 6.72981577e+01 6.92144629e+01 1.25701740e-02
 1.48168911e-07]
Uinsg R_m =  1.5
[3.92828758e+00 2.39594196e+01 1.34651399e+00 5.86119539e-06
 1.22323925e-09]
Uinsg R_m =  3
[8.08343980e+00 3.34259807e+01 2.34306375e-03 2.23023848e-05
 6.69599061e-09]
Uinsg R_m =  6
[4.43082576e+00 5.94270642e+00 5.66376379e-02 2.96933083e-04
 1.25822806e-05]
Uinsg R_m =  9
[1.45687346e+01 4.90683100e+01 1.93260827e-01 1.44316816e-02
 5.70982847e-03]
Uinsg R_m =  12
[22.65445675 95.98592289  1.81489453  0.27158335  2.39917298]


\begin{tabular}{c|ccc}
#Features $\backslash$ R & 1 & 2 & 3 \\
\hline
10 & 1.00e-02 & 8.00e-03 & 6.00e-03 \\
20 & 5.00e-03 & 4.00e-03 & 3.00e-03 \\
30 & 1.00e-03 & 9.00e-04 & 7.00e-04 \\
\end{tabular}


In [ ]:
R_m_list = [0.75, 1.5,3, 6,9,12]
num_diff_features_list = [ 100,200,400,800,1600]
table_latex = output_results_Rs_latex(err_list_Rs, R_m_list, num_diff_features_list)
print(table_latex)

\begin{tabular}{c|cccccc}
#Features $\backslash$ R & 0.75 & 1.5 & 3 & 6 & 9 & 12 \\
\hline
50 & 4.07e+00 & 3.93e+00 & 8.08e+00 & 4.43e+00 & 1.46e+01 & 2.27e+01 \\
100 & 6.73e+01 & 2.40e+01 & 3.34e+01 & 5.94e+00 & 4.91e+01 & 9.60e+01 \\
200 & 6.92e+01 & 1.35e+00 & 2.34e-03 & 5.66e-02 & 1.93e-01 & 1.81e+00 \\
400 & 1.26e-02 & 5.86e-06 & 2.23e-05 & 2.97e-04 & 1.44e-02 & 2.72e-01 \\
800 & 1.48e-07 & 1.22e-09 & 6.70e-09 & 1.26e-05 & 5.71e-03 & 2.40e+00 \\
\end{tabular}


### without partition of unity  

In [43]:
# analytical solution parameters
AA = 1
aa = 8.0*np.pi
bb = 8.0*np.pi
lamb = 4
activation = 'tanh'
k = 3 # if using tanh, k is not used 

def u(x):
    return AA * np.sin(bb * (x + 0.05)) * np.cos(aa * (x + 0.05)) + 2.0

def d2u_dx2(x):
    return -AA*(aa*aa+bb*bb)*np.sin(bb*(x+0.05))*np.cos(aa*(x+0.05))\
           -2.0*AA*aa*bb*np.cos(bb*(x+0.05))*np.sin(aa*(x+0.05))

def F(points, lamb):
    return(- d2u_dx2(points)  + lamb * u(points)) # change 


if __name__ == '__main__':
    lamb = 4
     
    J = 50 # the number of basis functions per PoU region
    Q = 50 # the number of collocation pointss per PoU region
    print_res = False  
    num_diff_features = 5 
    num_trials = 1 
    RFM_Error = np.zeros([num_diff_features,3]) 
    err_list_trials =  np.zeros([num_diff_features,num_trials])
    err_list_Rs = [] 
    num_diff_features_list =[] 
    activation = 'tanh'
    relu_k = 3 
    R_m_list = [3,6,9, 12,24,36,48]
    for R_m in R_m_list: 
        print("Uinsg R_m = ", R_m)
        print("========================================") 
        for i in range(num_diff_features): # the number of PoU regions
            for trial in range(num_trials):
                M_p = 1   
                J_n = J * (2**i)*2 
                Q = int(3200/M_p)
                RFM_Error[i,0] = int(M_p * J_n)
                models,w,RFM_Error[i,1], RFM_Error[i,2] = main(M_p,J_n,Q,lamb,activation=activation, k = relu_k,print_res= print_res)
                err_list_trials[i,trial] = RFM_Error[i,1] 
                
        print(err_list_trials.mean(axis=1)) 
        err_list_Rs.append(err_list_trials.mean(axis=1))

num_diff_features_list = [100,200,400,800,1600]
print(output_results_Rs_latex(err_list_Rs, R_m_list, num_diff_features_list))


Uinsg R_m =  3


/var/folders/45/p1_q2drx75983lk7ph_1gx6mhpr5_w/T/ipykernel_33385/4274200674.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return(math.sqrt((X_max-X_min)*sum(epsilon*epsilon)/len(epsilon)))


[ 2.49810708 18.27013454  6.72991383 12.93030021 11.02822333]
Uinsg R_m =  6
[ 4.98453594 34.88810885 10.47147251  8.15486468 14.12090768]
Uinsg R_m =  9
[  1.66667279   9.3073411   13.56640097 169.1080848    5.74606718]
Uinsg R_m =  12
[3.76247676e+00 2.13130162e+00 1.22514279e+01 1.11383672e-01
 3.85696051e-03]
Uinsg R_m =  24
[1.53228814e+01 5.93469549e+00 8.81739649e-02 5.27121220e-06
 3.32811026e-08]
Uinsg R_m =  36
[2.70658697e+01 1.65679066e+01 2.39545906e+00 4.12282883e-06
 3.83116520e-09]
Uinsg R_m =  48
[4.72590126e+01 2.86776475e+01 2.39839506e+01 3.21069949e-04
 1.29515249e-09]
\begin{tabular}{c|ccccccc}
#Features $\backslash$ R & 3 & 6 & 9 & 12 & 24 & 36 & 48 \\
\hline
100 & 2.50e+00 & 4.98e+00 & 1.67e+00 & 3.76e+00 & 1.53e+01 & 2.71e+01 & 4.73e+01 \\
200 & 1.83e+01 & 3.49e+01 & 9.31e+00 & 2.13e+00 & 5.93e+00 & 1.66e+01 & 2.87e+01 \\
400 & 6.73e+00 & 1.05e+01 & 1.36e+01 & 1.23e+01 & 8.82e-02 & 2.40e+00 & 2.40e+01 \\
800 & 1.29e+01 & 8.15e+00 & 1.69e+02 & 1.11e-01 & 5.27e-0

In [ ]:
print(output_results_Rs_latex(err_list_Rs_test, R_m_list, num_diff_features_list))

## multiscale frequency test - no pou 

In [ ]:
# analytical solution parameters
AA = 1
a1 = 2.0*np.pi
a2 = 8.0*np.pi
a3 = 16.0*np.pi
lamb = 4
activation = 'tanh'
k = 3 # if using tanh, k is not used 

def u(x):
    return AA * np.sin(a1 * x) + 0.5 * AA * np.sin(a2 * x) + 0.5 * AA * np.sin(a3 * x) + 2.0 

def d2u_dx2(x):
    return -AA*(a1*a1)*np.sin(a1*x) - 0.5*AA*(a2*a2)*np.sin(a2*x) - 0.5*AA*(a3*a3)*np.sin(a3*x)

def F(points, lamb):
    return(- d2u_dx2(points)  + lamb * u(points)) # change 


if __name__ == '__main__':
    lamb = 4
     
    J = 50 # the number of basis functions per PoU region
    Q = 50 # the number of collocation pointss per PoU region
    print_res = False  
    num_diff_features = 5 
    num_trials = 1 
    RFM_Error = np.zeros([num_diff_features,3]) 
    err_list_trials =  np.zeros([num_diff_features,num_trials])
    err_list_Rs = [] 
    num_diff_features_list =[] 
    activation = 'tanh'
    relu_k = 3 
    R_m_list = [3,6,9, 12,24,48]
    for R_m in R_m_list: 
        print("Uinsg R_m = ", R_m)
        print("========================================") 
        for i in range(num_diff_features): # the number of PoU regions
            for trial in range(num_trials):
                M_p = 1   
                J_n = J * (2**i)*2 
                Q = int(3200/M_p)
                RFM_Error[i,0] = int(M_p * J_n)
                models,w,RFM_Error[i,1], RFM_Error[i,2] = main(M_p,J_n,Q,lamb,activation=activation, k = relu_k,print_res= print_res)
                err_list_trials[i,trial] = RFM_Error[i,1] 
                
        print(err_list_trials.mean(axis=1)) 
        err_list_Rs.append(err_list_trials.mean(axis=1))

num_diff_features_list = [100,200,400,800,1600]
print(output_results_Rs_latex(err_list_Rs, R_m_list, num_diff_features_list))

Uinsg R_m =  3


/var/folders/45/p1_q2drx75983lk7ph_1gx6mhpr5_w/T/ipykernel_33385/4274200674.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return(math.sqrt((X_max-X_min)*sum(epsilon*epsilon)/len(epsilon)))


[ 1.56869826 16.72712953  5.31966796  3.8324743   6.60723087]
Uinsg R_m =  6
[15.15987074 12.86179818  2.98136985  3.35195985 11.21995062]
Uinsg R_m =  9
[20.74379925  3.18224963 69.02755082 67.8830818   8.14286291]
Uinsg R_m =  12
[7.20025318e+00 7.52716986e+01 1.18714323e+01 4.99621971e-02
 6.02504010e-03]
Uinsg R_m =  24
[1.03111382e+01 1.30539365e+01 1.17959337e+00 7.39353726e-06
 1.43091053e-07]
Uinsg R_m =  48
[5.94295109e+01 4.41978453e+01 9.57468873e-01 2.76463775e-04
 2.55326840e-08]
\begin{tabular}{c|cccccc}
#Features $\backslash$ R & 3 & 6 & 9 & 12 & 24 & 48 \\
\hline
100 & 1.57e+00 & 1.52e+01 & 2.07e+01 & 7.20e+00 & 1.03e+01 & 5.94e+01 \\
200 & 1.67e+01 & 1.29e+01 & 3.18e+00 & 7.53e+01 & 1.31e+01 & 4.42e+01 \\
400 & 5.32e+00 & 2.98e+00 & 6.90e+01 & 1.19e+01 & 1.18e+00 & 9.57e-01 \\
800 & 3.83e+00 & 3.35e+00 & 6.79e+01 & 5.00e-02 & 7.39e-06 & 2.76e-04 \\
1600 & 6.61e+00 & 1.12e+01 & 8.14e+00 & 6.03e-03 & 1.43e-07 & 2.55e-08 \\
\end{tabular}


In [41]:
err_list_Rs
R_m_list
num_diff_features_list = [100,200]
print(output_results_Rs_latex(err_list_Rs, R_m_list, num_diff_features_list))

\begin{tabular}{c|cc}
#Features $\backslash$ R & 3 & 6 \\
\hline
100 & 7.57e+00 & 2.94e+00 \\
200 & 3.01e+00 & 5.40e+00 \\
\end{tabular}


IndexError: index 3 is out of bounds for axis 0 with size 3

In [48]:
M_p = 5 
Q = 10 
points = []
for n in range(M_p):
    x_min = (X_max-X_min)/M_p * n + X_min
    x_max = (X_max-X_min)/M_p * (n+1) + X_min
    points.append(np.linspace(x_min, x_max, Q+1).reshape([-1,1]))

In [50]:
points[0].shape  

(11, 1)